# 04 — Modeling

Trains two classifiers on the cleaned dataset and saves the best model.

**Inputs:**
- `data/train.parquet` / `data/test.parquet` (from `02_data_cleaning.ipynb`)
- `models/label_encoder.pkl`

**Models trained:**
1. **TF-IDF + LinearSVC** — trained on a stratified 1M-row subsample (fast, calibrated probabilities)
2. **FastText** — trained on the full dataset (~5.5M rows), streams text so RAM stays low

**Outputs:**
- `models/model_svc.pkl`
- `models/model_fasttext.bin`
- `data/train_ft.txt` / `data/test_ft.txt` (FastText input format)
- `evaluation/final_report.json`

In [ ]:
import os, time, warnings, json
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import f1_score, classification_report
warnings.filterwarnings('ignore')

PROJ = '/home/tayana-gpu/ML/project_01_product_classifier'

print('Loading train/test data...')
train_df = pd.read_parquet(f'{PROJ}/data/train.parquet')
test_df  = pd.read_parquet(f'{PROJ}/data/test.parquet')
le       = joblib.load(f'{PROJ}/models/label_encoder.pkl')

X_test = test_df['text'].tolist()
y_test = test_df['label'].tolist()
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}  |  Classes: {len(le.classes_)}')

In [ ]:
# Model 1: TF-IDF + LinearSVC on 1M stratified subsample
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split

print('=' * 60)
print('MODEL 1: TF-IDF + LinearSVC (1M subsample)')
print('=' * 60)

SVC_SAMPLE = 1_000_000
if len(train_df) > SVC_SAMPLE:
    print(f'Subsampling train to {SVC_SAMPLE:,} rows (stratified)...')
    svc_train, _ = train_test_split(
        train_df, train_size=SVC_SAMPLE,
        stratify=train_df['label'], random_state=42
    )
else:
    svc_train = train_df

X_train_svc = svc_train['text'].tolist()
y_train_svc = svc_train['label'].tolist()
print(f'SVC training rows: {len(X_train_svc):,}')

word_vec = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2),
    min_df=5, max_features=150_000, sublinear_tf=True
)
char_vec = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5),
    min_df=5, max_features=100_000, sublinear_tf=True
)

pipeline_svc = Pipeline([
    ('features', FeatureUnion([('word', word_vec), ('char', char_vec)])),
    ('clf', CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', max_iter=2000, C=1.0),
        cv=3, method='isotonic', n_jobs=3
    ))
])

t0 = time.time()
print('Training...')
pipeline_svc.fit(X_train_svc, y_train_svc)
y_pred_svc   = pipeline_svc.predict(X_test)
svc_macro    = f1_score(y_test, y_pred_svc, average='macro')
svc_weighted = f1_score(y_test, y_pred_svc, average='weighted')
print(f'Done in {time.time()-t0:.0f}s')
print(f'Macro F1: {svc_macro:.4f}  |  Weighted F1: {svc_weighted:.4f}')
print(classification_report(y_test, y_pred_svc, target_names=le.classes_))

joblib.dump(pipeline_svc, f'{PROJ}/models/model_svc.pkl')
print('Saved models/model_svc.pkl')

In [ ]:
# Model 2: FastText on the full training set
import fasttext

print('=' * 60)
print('MODEL 2: FastText (full training set)')
print('=' * 60)

def write_ft_file(texts, labels, path, le):
    with open(path, 'w', encoding='utf-8') as f:
        for text, label in zip(texts, labels):
            label_str = le.classes_[label].replace(' ', '_')
            f.write(f'__label__{label_str} {text}\n')

X_train_ft = train_df['text'].tolist()
y_train_ft = train_df['label'].tolist()

train_ft = f'{PROJ}/data/train_ft.txt'
test_ft  = f'{PROJ}/data/test_ft.txt'
print(f'Writing FastText files ({len(X_train_ft):,} train rows)...')
write_ft_file(X_train_ft, y_train_ft, train_ft, le)
write_ft_file(X_test,     y_test,     test_ft,  le)
print('Done writing.')

t0 = time.time()
print('Training FastText...')
ft_model = fasttext.train_supervised(
    input=train_ft,
    lr=0.5, epoch=15, wordNgrams=2,
    dim=100, minCount=5,
    loss='softmax', thread=8
)
print(f'Done in {time.time()-t0:.0f}s')

result      = ft_model.test(test_ft)
ft_preds    = [ft_model.predict(t)[0][0].replace('__label__', '').replace('_', ' ') for t in X_test]
ft_true     = [le.classes_[l] for l in y_test]
ft_macro    = f1_score(ft_true, ft_preds, average='macro')
ft_weighted = f1_score(ft_true, ft_preds, average='weighted')
print(f'Samples={result[0]}  P={result[1]:.4f}  R={result[2]:.4f}')
print(f'Macro F1: {ft_macro:.4f}  |  Weighted F1: {ft_weighted:.4f}')
print(classification_report(ft_true, ft_preds))

ft_model.save_model(f'{PROJ}/models/model_fasttext.bin')
print('Saved models/model_fasttext.bin')

In [ ]:
# Final comparison and save evaluation report
print('=' * 60)
print('FINAL COMPARISON')
print('=' * 60)
print(f'  LinearSVC (1M sample, TF-IDF):  Macro F1 = {svc_macro:.4f}')
print(f'  FastText  (full dataset, 15ep):  Macro F1 = {ft_macro:.4f}')

best_name  = 'LinearSVC' if svc_macro >= ft_macro else 'FastText'
best_macro = max(svc_macro, ft_macro)
print(f'\nBest: {best_name}  (Macro F1 = {best_macro:.4f})')

os.makedirs(f'{PROJ}/evaluation', exist_ok=True)
if best_name == 'LinearSVC':
    best_true  = [le.classes_[l] for l in y_test]
    best_preds = y_pred_svc
else:
    best_true  = ft_true
    best_preds = ft_preds

report = classification_report(best_true, best_preds, output_dict=True)
with open(f'{PROJ}/evaluation/final_report.json', 'w') as f:
    json.dump({'best_model': best_name, 'svc_macro': svc_macro,
               'ft_macro': ft_macro, 'report': report}, f, indent=2)

print('\nSaved evaluation/final_report.json')
print('\nDONE.')